# Phase 1 Report
## Models
- TF-IDF
- BM25
- Embedding

For this project, two methods were implemented independently, but also simultaneously, from 2 duos, before joining officially as the group 'pain au chocolat'.  
Thus we implemented separately, for what we consider the most interesting implementations :
- the TF-IDF and BM25+ methods
- the embedding method

Each of these methods are presented independently here.

# PART 1 : TF-IDF and BM25+ METHODS

### ***Load and prepare the data***

In [ ]:
# Load and prepare the data

import json
with open("retrieval-engine-competition/docs.json", "r") as file:
    docs = json.load(file)
with open("retrieval-engine-competition/queries_train.json", "r") as file:
    queries = json.load(file)
with open("retrieval-engine-competition/qgts_train.json", "r") as file:
    relevant_GT = json.load(file)

print(f"Number of documents : {len(docs)}")
print(f"Number of queries : {len(queries)}")
print(f"Number of queries with relevant ground truth : {len(relevant_GT)}")

Statistics for the documents :

In [ ]:
import numpy as np
doc_text_lengths = [len(doc["text"]) for doc in docs]
doc_title_lengths = [len(doc["title"]) for doc in docs]

# Compute the unique categories and tags in the 'docs' file, and their number of occurence
unique_doc_categories = {}
unique_doc_tags = {}

for doc in docs:
    if doc["category"] not in unique_doc_categories:
        unique_doc_categories[doc["category"]] = 0
    unique_doc_categories[doc["category"]] += 1
    for tag in doc["tags"]:
        if tag not in unique_doc_tags:
            unique_doc_tags[tag] = 0
        unique_doc_tags[tag] += 1


print("Document 'text' length stats : mean = {:.1f}, std = {:.1f}, min = {}, max = {}".format(np.mean(doc_text_lengths), np.std(doc_text_lengths), np.min(doc_text_lengths), np.max(doc_text_lengths)))
print("\t -> nb of empty 'text' : {}".format(len([length for length in doc_text_lengths if length == 0])))
print("Document 'title' length stats : mean = {:.1f}, std: {:.1f}, min = {}, max = {}".format(np.mean(doc_title_lengths), np.std(doc_title_lengths), np.min(doc_title_lengths), np.max(doc_title_lengths)))
print("Document 'tags' stats : nb unique tags = {}, min occurences for a tag = {}, and max = {}".format(len(unique_doc_tags), np.min(list(unique_doc_tags.values())), np.max(list(unique_doc_tags.values()))))
print("Document 'category' stats : nb unique categories = {}, min occurences of a category = {}, and max = {}".format(len(unique_doc_categories), np.min(list(unique_doc_categories.values())), np.max(list(unique_doc_categories.values()))))

Statistics for the queries :

In [ ]:
queries_text_lengths = [len(query["text"]) for query in queries]

# Compute the unique categories and tags in the 'queries' file, and their number of occurence
unique_query_categories = {}
unique_query_tags = {}

for query in queries:
    if query["category"] not in unique_query_categories:
        unique_query_categories[query["category"]] = 0
    unique_query_categories[query["category"]] += 1
    for tag in query["tags"]:
        if tag not in unique_query_tags:
            unique_query_tags[tag] = 0
        unique_query_tags[tag] += 1

print("Queries 'text' length stats : mean = {:.1f}, std = {:.1f}, min = {}, max = {}".format(np.mean(queries_text_lengths), np.std(queries_text_lengths), np.min(queries_text_lengths), np.max(queries_text_lengths)))
print("Queries 'title' stats : nb of titles not empty = {}".format(len([query["title"] for query in queries if len(query["title"]) != 0])))
print("Queries 'tags' stats : nb unique tags = {}, min occurences for a tag = {}, and max = {}".format(len(unique_query_tags), np.min(list(unique_query_tags.values())), np.max(list(unique_query_tags.values()))))
print("\t -> are all those tags in the 'docs' file too ? {}".format(set(unique_query_tags).issubset(set(unique_doc_tags))))
print("Queries 'category' stats : nb unique categories = {}, min occurences for a category = {}, and max = {}".format(len(unique_query_categories), np.min(list(unique_query_categories.values())), np.max(list(unique_query_categories.values()))))
print("\t -> are all those categories the same than those in the 'docs' file ? {}".format( set(unique_query_categories) == set(unique_doc_categories) ))


Statistics for the relevant ground truth :

In [ ]:
relevantGT_nbRelevantDocs = [ relevant_GT[id]["total_relevant_docs"] for id in relevant_GT]

relevance_values = []
for id in relevant_GT:
    for rel_doc in relevant_GT[id]["relevant_doc_ids"]:
        relevance_values.append(rel_doc["relevance"])


print("Relevant GT 'nb_relevant_docs' stats : mean = {:.1f}, std = {:.1f}, min = {}, max = {}".format(np.mean(relevantGT_nbRelevantDocs), np.std(relevantGT_nbRelevantDocs), np.min(relevantGT_nbRelevantDocs), np.max(relevantGT_nbRelevantDocs)))
print("Relevant GT 'relevance' values : {}".format(np.unique(relevance_values)))

- *imports for the TF-IDF and BM25+ methods :*

In [ ]:
!pip install contractions
!pip install rank_bm25 scikit-learn
!pip install faiss-cpu --quiet

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Plus
import umap
import matplotlib.pyplot as plt
import faiss
from sentence_transformers import SentenceTransformer
from pathlib import Path
import pandas as pd
import time
from pathlib import Path
from collections import Counter
import numpy as np
import json
import re
import string
import contractions
import csv
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from tqdm import tqdm
import nltk
from nltk.tokenize import word_tokenize
import random
from nltk.corpus import wordnet
from nltk.stem import PorterStemmer
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')
nltk.download('omw-1.4')

# **Text Preprocessing Techniques**
The 'content'  field is constructed using several techniques like:

* Contraction Expansion.

* Lowercasing.

* Punctuation Removal.

* Stopword Removal.*texte en italique*

* Token Normalization & Stemming.

These techniques are described more precisly in the report.

In [ ]:
# Initialisation
stop_words = set(stopwords.words('english'))
#lemmatizer = WordNetLemmatizer()
stemmer = PorterStemmer()

def full_process(text):
    if not text: return ""
    # Contractions (don't -> do not)
    text = contractions.fix(text)
    # Lowercasing
    text = text.lower()
    # Remove Punctuation & Special Chars
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    # Tokenization & Stopwords & Stemmatization
    tokens = word_tokenize(text)
    #cleaned = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    cleaned = [stemmer.stem(t) for t in tokens if t not in stop_words]
    return " ".join(cleaned)

print("docs processing...")
for d in tqdm(docs):
    # RAW field (for Embeddings, because he don't really need it)
    d["content_raw"] = f"{d.get('title','') or ''} {d.get('text','') or ''} {' '.join(d.get('tags',[]) or [])}".strip()
     # Cleanin field (for BM25/TF-IDF)
    forcleanin = f"{d.get('title','') or ''} {d.get('title','') or ''} {d.get('text','') or ''} {' '.join(d.get('tags',[]) or [])}".strip()
    d["content_clean"] = full_process(forcleanin)

# **Query Preparation**

Prepares queries for lexical retrieval (TF-IDF, BM25+) by applying cleaning and controlled synonym expansion.

- Cleaning: uses `full_process()`
- Expansion: WordNet-based, applied only to short queries (≤ 8 words)
- Synonyms added with 30% probability to reduce vocabulary mismatch

In [ ]:
class SynonymExpander:
    """Expand query tokens using WordNet synonyms.
    Args:
        add_synonym_prob (float): Probability of adding the most common
                                  synonym for a given word (default=0.3).
    """
    def __init__(self, add_synonym_prob=0.3):
        self.add_synonym_prob = add_synonym_prob

    def expand(self, tokens):
      """
      Expand a list of tokens with WordNet synonyms.
      Args:
          tokens (List[str]): Tokenized query words.
      Returns:
          Original tokens plus selected synonyms.
      """
      expanded = []
      for word in tokens:
        synsets = wordnet.synsets(word)
        if synsets and random.random() <= self.add_synonym_prob:
          # Retrieve the most commun synonym
          syn_word = synsets[0].lemmas()[0].name().lower().replace('_', ' ')
          if syn_word != word:
            expanded.append(syn_word)
      return tokens + expanded

expander = SynonymExpander(add_synonym_prob=0.3)

def prepare_query(query_obj):
    """Prepare and expand a query for lexical retrieval models.
    This function:
        - Extracts text and tags
        - Applies token filtering for WordNet compatibility
        - Performs synonym expansion for short queries (≤ 8 words)
        - Applies the full preprocessing pipeline

    Args:
        query_obj (dict): Query object containing 'title', 'text', and 'tags'.

    Returns:
        str: Fully processed query string ready for TF-IDF and BM25+.
    """
    text = query_obj.get('text', '') or ''
    tags = " ".join(query_obj.get('tags', []) or [])

    base_text = f"{text}".strip()

    # Tokenization and cleanin gor WordNet recognize words
    temp_tokens = word_tokenize(base_text.lower())
    temp_tokens = [t for t in temp_tokens if t.isalpha() and t not in stop_words]

    # Expansion only if the query is short (under 8 words)
    expanded_text = ""
    if len(temp_tokens) <= 8:
        # We use expander here
        synonyms = expander.expand(temp_tokens)
        expanded_text = " ".join(synonyms)

    combined = f"{text} {tags} {expanded_text}".strip()

    return full_process(combined)

# **TF-IDF and BM25+ initialization**

In [ ]:
# Gloabal list preparation
print("build list of IDs...")
doc_ids = [d["id"] for d in docs]
corpus_clean = [d["content_clean"] for d in docs]

# TF-IDF Initialization
print("build matrice for TF-IDF...")
vectorizer = TfidfVectorizer(norm='l2')
tfidf_matrix = vectorizer.fit_transform(corpus_clean)

# BM25+ Initialization
print("build index for BM25+...")
tokenized_corpus = [doc.split() for doc in corpus_clean]
bm25_model = BM25Plus(tokenized_corpus, b=0.75)

print("OK !")

# **TF-IDF Implementation**

In [ ]:
def run_tfidf(query_obj, k=10):
    """Retrieve the top-k most relevant documents using TF-IDF.

    This function:
        - Prepares the query using the lexical preprocessing pipeline
        - Transforms the query into a TF-IDF vector
        - Computes similarity scores between the query and all documents
        - Returns the top-k ranked document IDs

    Args:
        query_obj (dict): Query object containing 'title', 'text', and 'tags'.
        k (int): Number of top documents to retrieve.

    Returns:
        List[str]: IDs of the top-k most similar documents.
    """

    query_clean = prepare_query(query_obj)

    # Transform query as vector
    query_vec = vectorizer.transform([query_clean])

    # Calculation of similarity scores
    scores = (tfidf_matrix * query_vec.T).toarray().flatten()

    # Retreive the K best docs
    best_indices = np.argsort(-scores)[:k]
    return [doc_ids[i] for i in best_indices]

# **BM25+ Implementation**

In [ ]:
def run_bm25(query_obj, k=10):
    """Retrieve the top-k most relevant documents using BM25+.

    This function:
        - Prepares and tokenizes the query
        - Computes BM25+ relevance scores for all documents
        - Ranks documents based on their scores
        - Returns the top-k ranked document IDs

    Args:
        query_obj (dict): Query object containing 'title', 'text', and 'tags'.
        k (int): Number of top documents to retrieve.

    Returns:
        List[str]: IDs of the top-k most relevant documents.
    """
    # Preprocess and tokenize the query
    query_clean = prepare_query(query_obj)
    query_tokens = query_clean.split()

    # Compute BM25+ scores for all documents
    scores = bm25_model.get_scores(query_tokens)

    # Ensure k is not larger than the number of documents
    k = min(k, len(scores))

    # Get indices of the top-k scores
    top_k_idx = np.argpartition(-scores, k-1)[:k]

    # Sort the top-k indices by score descending
    top_k_idx = top_k_idx[np.argsort(-scores[top_k_idx])]

    # Return the corresponding document IDs
    return [doc_ids[i] for i in top_k_idx]

# **Evaluation of TF-IDF & BM25+**

This section evaluates the retrieval models (TF-IDF and BM25+) using standard Information Retrieval metrics.

Metrics computed per query:
- Precision
- Recall
- Mean Reciprocal Rank (MRR)

The system also measures the average latency (response time per query).


In [ ]:
def calculate_query_metrics(relevant_ids, retrieved_ids):
    """
    Compute evaluation metrics for a single query.
    Metrics computed:
        - Precision
        - Recall
        - Mean Reciprocal Rank (MRR)

    Args:
        relevant_ids (List[str]): Ground-truth relevant document IDs.
        retrieved_ids (List[str]): Retrieved document IDs.

    Returns:
        Tuple[float, float, float]: (precision, recall, mrr)
    """
    relevant = set([str(rid) for rid in relevant_ids])
    retrieved = [str(rid) for rid in retrieved_ids]
    k = len(retrieved)

    if not relevant: return 0, 0, 0

    # Intersection : Documents bien trouvés
    hits = [1 if doc_id in relevant else 0 for doc_id in retrieved]
    num_hits = sum(hits)

    # Precision = Corrects / Total retrieve (k)
    precision = num_hits / k

    # Recall = Corrects / Total relevant
    recall = num_hits / len(relevant)

    # MRR = 1 / range of the first correct document
    mrr = 0
    for i, hit in enumerate(hits):
        if hit:
            mrr = 1 / (i + 1)
            break

    return precision, recall, mrr

def evaluate_retrieval_system(queries, qgts, retrieval_func, k=10):
    """Evaluate a retrieval method over a set of queries.

    This function:
        - Applies the retrieval function to each query
        - Computes Precision, Recall, and MRR
        - Measures average latency per query

    Args:
        queries (List[dict]): List of query objects.
        qgts (dict): Ground-truth relevance judgments.
        retrieval_func (callable): Retrieval function (e.g., TF-IDF or BM25+).
        k (int): Number of top documents to retrieve.

    Returns:
        dict: Dictionary containing average precision, recall,
              MRR, and latency.
    """
    all_p, all_r, all_mrr = [], [], []
    start_time = time.time()

    for query_obj in queries:
        query_id = query_obj['id']

        # Relevant docs
        relevant_data = qgts.get(query_id, {}).get('relevant_doc_ids', [])
        relevant_ids = []
        for item in relevant_data:
            if isinstance(item, dict):
                relevant_ids.append(str(item.get('id', item.get('doc_id'))))
            else:
                relevant_ids.append(str(item))

        # Retrieval ids
        retrieved_ids = retrieval_func(query_obj, k=k)

        # Calcul metric/query
        p, r, mrr = calculate_query_metrics(relevant_ids, retrieved_ids)
        all_p.append(p)
        all_r.append(r)
        all_mrr.append(mrr)

    end_time = time.time()
    latency = (end_time - start_time) / len(queries) # time/query

    return {
        "avg_precision": np.mean(all_p),
        "avg_recall": np.mean(all_r),
        "avg_mrr": np.mean(all_mrr),
        "avg_latency": latency
    }

methods = {
    "TF-IDF": run_tfidf,
    "BM25+": run_bm25
}

results = {}
for name, func in methods.items():
    print(f"Évaluation de {name}...")
    results[name] = evaluate_retrieval_system(queries, relevant_GT, retrieval_func=func, k=20)

# Display
print("\n" + "="*50)
print(f"{'Modèle':<12} | {'P@10':<8} | {'R@10':<8} | {'MRR':<8} | {'Latence (s)':<8}")
print("-" * 50)
for name, m in results.items():
    print(f"{name:<12} | {m['avg_precision']:.4f} | {m['avg_recall']:.4f} | {m['avg_mrr']:.4f} | {m['avg_latency']:.4f}")

# PART 2 : EMBEDDING METHOD

*Question : What is the purpose of high-dimensional embeddings? How can they help in information retrieval compared to simpler models?*

Embedding models translate some text (or some object) into a vector of numeric values. I suppose those values are floats in the normalized range of [-1 ; 1], because they are easier to work with for most models.  
Thus, embeddings are useful notably to compare two data via their embeddings, in particular assuming the embedding vectors are the same size (same number of dimensions).  
Finally, embeddings store informations about the original data using several values, consisting of the dimensions of the embedding. The more informations there are, the more precise the embedding is in regard to the original data. This is why embedding vectors usually have hundreds or thousands of values (so are of hundreds / thousands of dimensions).

## Data Loading and Text Preprocessing For Embeddings

This code block loads the dataset and prepares the textual input for retrieval models.

First, the document collection, training queries, test queries, and ground-truth relevance labels are loaded from JSON files and converted into structured DataFrames.

Then, textual information from different fields (title, text, and tags) is merged into a single unified representation called *content*. Missing values are handled and tags are normalized to ensure consistent formatting.

In [ ]:
# init
# PROJECT_ROOT = Path.cwd().parent
# DATA_DIR = PROJECT_ROOT / "retrieval-engine-competition"

# print("PROJECT_ROOT:", PROJECT_ROOT)
# print("DATA_DIR exists?", DATA_DIR.exists())

DATA_DIR = Path("/content/retrieval-engine-competition")

print(f"DATA_DIR exists? ? {DATA_DIR.exists()}")

def load_json(p: Path):
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

docs = load_json(DATA_DIR / "docs.json")
qtrain = load_json(DATA_DIR / "queries_train.json")
qtest = load_json(DATA_DIR / "queries_test.json")
gts = load_json(DATA_DIR / "qgts_train.json")

docs_df = pd.DataFrame(docs)
qtrain_df = pd.DataFrame(qtrain)
qtest_df = pd.DataFrame(qtest)

def norm_tags(tags):
    if tags is None: return []
    if isinstance(tags, list): return [str(x) for x in tags]
    return []

def build_doc_content(row):
    title = "" if row.get("title") is None else str(row.get("title"))
    text  = "" if row.get("text")  is None else str(row.get("text"))
    tags  = " ".join(norm_tags(row.get("tags")))
    return " ".join([title, text, tags]).strip()

def build_query_content(row):
    title = "" if row.get("title") is None else str(row.get("title"))
    text  = "" if row.get("text")  is None else str(row.get("text"))
    tags  = " ".join(norm_tags(row.get("tags")))
    return " ".join([title, text, tags]).strip()

docs_df["content"] = docs_df.apply(build_doc_content, axis=1)
qtrain_df["content"] = qtrain_df.apply(build_query_content, axis=1)
qtest_df["content"]  = qtest_df.apply(build_query_content, axis=1)

print("Loaded ✅", len(docs_df), len(qtrain_df), len(qtest_df), len(gts))


## Evaluation Procedure For Embeddings

This code block implements the evaluation framework for the retrieval system.

First, it extracts the ground-truth relevant document IDs for each query. The function handles different possible formats of relevance labels to ensure robustness and consistency.

Then, it evaluates the retrieval results using three standard metrics:

- **Recall@k**: proportion of relevant documents retrieved within the top-k results  
- **Precision@k**: proportion of retrieved documents that are relevant  
- **MRR (Mean Reciprocal Rank)**: measures how early the first relevant document appears in the ranking  

For each query, the system compares the top-k retrieved documents with the ground-truth relevant set. The metrics are computed per query and averaged across all queries to obtain the final performance scores.

In [ ]:
def get_gt_docids(gts, qid):
    item = gts.get(str(qid), None)
    if item is None:
        return []

    rels = item.get("relevant_doc_ids", [])

    if isinstance(rels, list) and len(rels) > 0 and isinstance(rels[0], dict):
        return [str(x["doc_id"]) for x in rels if "doc_id" in x]
    return list(map(str, rels))


def evaluate(results, gts, k):
    recalls, precisions, mrrs = [], [], []
    for qid, retrieved in results.items():
        gt = set(get_gt_docids(gts, qid))
        retrieved = list(map(str, retrieved[:k]))

        hits = [1 if d in gt else 0 for d in retrieved]
        r = sum(hits) / len(gt) if len(gt) else 0.0
        p = sum(hits) / k if k else 0.0

        rr = 0.0
        for rank, h in enumerate(hits, start=1):
            if h:
                rr = 1.0 / rank
                break

        recalls.append(r); precisions.append(p); mrrs.append(rr)

    return float(np.mean(recalls)), float(np.mean(precisions)), float(np.mean(mrrs))


## Generate embeddings using SentenceTransformers

This code block generates dense vector representations for documents and queries using a pre-trained SentenceTransformer model.

All textual content is encoded into fixed-length numerical vectors that capture semantic meaning. Embeddings are normalized to unit length so that cosine similarity can be computed efficiently using vector operations.

The resulting document and query embeddings serve as the representation space for semantic retrieval, enabling comparison based on meaning rather than exact word matching.

In [ ]:
model_name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(model_name)
print("Loaded model:", model_name)
docs_texts = docs_df["content"].fillna("").tolist()
qtrain_texts = qtrain_df["content"].fillna("").tolist()

# This part of code is commented out because it may take a very long time to run.
# Execute the cell below to load the results from saved files instead.

# doc_emb = model.encode(
#     docs_texts,
#     batch_size=256,
#     show_progress_bar=True,
#     convert_to_numpy=True,
#     normalize_embeddings=True,
# )

# qry_emb = model.encode(
#     qtrain_texts,
#     batch_size=256,
#     show_progress_bar=True,
#     convert_to_numpy=True,
#     normalize_embeddings=True,
# )

# print("doc_emb:", doc_emb.shape, doc_emb.dtype)
# print("qry_emb:", qry_emb.shape, qry_emb.dtype)

In [ ]:
# np.save("doc_emb.npy", doc_emb)
# np.save("qry_emb.npy", qry_emb)
doc_emb = np.load("doc_emb.npy")
qry_emb = np.load("qry_emb.npy")
print("Document embeddings shape:", doc_emb.shape)
print("Query embeddings shape:", qry_emb.shape)

### Visualizing Document Embeddings with UMAP

This cell samples 2,000 documents and reduces their high-dimensional embeddings (384 dimensions) to 2 dimensions using UMAP.  

The goal is to visualize relationships between documents in 2D, allowing us to observe natural structures or clusters in the data. Each point represents a document, and its position reflects semantic similarity to other documents.

In [ ]:
import umap
import matplotlib.pyplot as plt

# Sample a subset for faster UMAP visualization
np.random.seed(42)
n = 2000
idx = np.random.choice(len(doc_emb), size=n, replace=False)
X = doc_emb[idx]

# UMAP projection
reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, metric="cosine", random_state=42)
X2 = reducer.fit_transform(X)

# Plot
plt.figure(figsize=(10, 7))
plt.scatter(X2[:, 0], X2[:, 1], s=8, alpha=0.7, color='blue')
plt.title("UMAP of Document Embeddings")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.tight_layout()
plt.savefig("umap_document_embeddings.png", dpi=300, bbox_inches="tight")
plt.show()

### Observations from 2D Visualization

- Even without using categories, embeddings show some natural clustering.  
- Documents with similar content tend to be close together because embeddings capture semantic similarity.  
- This demonstrates that embeddings can capture latent structure, which is useful for semantic search.

## Embedding-Based Retrieval with FAISS

This code block performs similarity search in the embedding space using FAISS.

A vector index is constructed from the document embeddings to enable efficient large-scale similarity search. Each query embedding is then compared against all document embeddings using inner product similarity.

For each query, the top-k most similar documents are retrieved based on their embedding similarity scores.

This approach enables fast and scalable semantic retrieval in high-dimensional vector space.

In [ ]:
d = doc_emb.shape[1]
index = faiss.IndexFlatIP(d)  # Using Inner Product = cosine similarity if embeddings are normalized
index.add(doc_emb)

k = 20
scores, idxs = index.search(qry_emb, k)  # idxs shape: [#queries, k]

print("Top-k indices shape:", idxs.shape)
print("Top-k scores shape:", scores.shape)

# Map query IDs to top-k document IDs
doc_ids = docs_df["id"].astype(str).tolist()
q_ids = qtrain_df["id"].astype(str).tolist()

topk_results = {str(qid): [doc_ids[j] for j in idxs[i]] for i, qid in enumerate(q_ids)}

## Embedding Retrieval Results and Evaluation

This code block converts the FAISS search results into document IDs and evaluates the embedding-based retrieval model.

For each query, the indices returned by FAISS are mapped back to their corresponding document IDs. The top-k retrieved documents are then compared with the ground-truth relevant documents.

The performance is measured using Recall@k, Precision@k, and Mean Reciprocal Rank (MRR).

This step provides the final evaluation of the dense retrieval model and allows direct comparison with TF-IDF and BM25.

In [ ]:
doc_ids = docs_df["id"].astype(str).tolist()
q_ids = qtrain_df["id"].astype(str).tolist()

emb_results = {}
for qi, qid in enumerate(q_ids):
    emb_results[str(qid)] = [doc_ids[j] for j in idxs[qi]]

recall, precision, mrr = evaluate(emb_results, gts, k)
print(f"Embeddings(all-MiniLM-L6-v2) @k={k}: recall={recall:.4f}, precision={precision:.4f}, mrr={mrr:.4f}")

## Test Query Embedding Generation

This code block generates dense vector representations for the test queries using the same pre-trained SentenceTransformer model.

The test query texts are encoded into normalized embedding vectors, consistent with the document and training query representations.

These embeddings are used to perform semantic retrieval on unseen queries, enabling evaluation on the Kaggle test set.

In [ ]:
qtest_texts = qtest_df["content"].fillna("").tolist()

qtest_emb = model.encode(
    qtest_texts,
    batch_size=256,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

## Kaggle Submission Generation

This code block generates the final retrieval results for the test queries and formats them for Kaggle submission.

For each test query, similarity scores are computed between the query embedding and all document embeddings. The top-k most similar documents are selected as the predicted relevant documents.

The results are formatted according to the competition requirements and exported as a CSV file containing:

- query_id
- retrieved document IDs  
- category empty

This file is submitted to Kaggle for evaluation on the hidden test set.

In [ ]:
k = 20

# IDs documents
doc_ids = docs_df["id"].astype(str).tolist()
# IDs queries test
query_ids = qtest_df["id"].astype(str).tolist()

scores = np.dot(qtest_emb, doc_emb.T)   # (nb_queries, nb_docs)

# Top-k indices
topk_indices = np.argsort(-scores, axis=1)[:, :k]
rows = []
for i, qid in enumerate(query_ids):
    top_docs = [doc_ids[idx] for idx in topk_indices[i]]

    rows.append({
        "query_id": qid,
        "relevant_doc_ids": json.dumps(top_docs),
        "category": '""'
    })

sub = pd.DataFrame(rows, columns=["query_id", "relevant_doc_ids", "category"])
sub.to_csv(
    "submission_phase1.csv",
    index=False,
    quoting=csv.QUOTE_ALL
)
print("saved submission_phase1.csv, shape=", sub.shape)
print(sub.head(2))

## Conclusion

In this project, we implemented a complete information retrieval pipeline, including data preprocessing, lexical retrieval models (TF-IDF and BM25+), and semantic retrieval using dense embeddings.

All methods were evaluated using standard metrics, including Recall, Precision, and Mean Reciprocal Rank. Experimental results showed that embedding-based retrieval significantly outperformed traditional lexical methods, demonstrating the advantage of semantic representations in capturing textual meaning.

We also analyzed the structure of the embedding space through UMAP visualization and observed meaningful clustering across document categories.

Finally, the system was applied to unseen test queries, and retrieval results were formatted for Kaggle submission, completing an end-to-end retrieval workflow.

Overall, this project demonstrates the effectiveness of semantic retrieval methods and provides a solid foundation for further improvements in Phase 2.